In [ ]:
!pip install apify-client

In [ ]:
from apify_client import ApifyClient

client = ApifyClient("MASUKKAN_TOKEN_DISINI")

# simpan ke variabel
run_input = {
    "commentsPerPost": 150,
    "excludePinnedPosts": False,
    "maxRepliesPerComment": 5,
    "postURLs": [
        "https://www.tiktok.com/@maswowogemoy/video/7606259380924812564",
        "https://www.tiktok.com/@irwandiferry/video/7553959113705721096",
        "https://www.tiktok.com/@watchdoc_id/video/7615597115150060821",
        "https://www.tiktok.com/@triasambari/video/7616346963302157576",
        "https://www.tiktok.com/@pojoksatu.id/video/7614019354400738578"
    ],
    "resultsPerPage": 100
}

run = client.actor("BDec00yAmCm1QbMEI").call(run_input=run_input)

items = list(client.dataset(run["defaultDatasetId"]).iterate_items())

In [ ]:
import pandas as pd
df = pd.DataFrame(items)
df.head()

In [ ]:
df.to_excel('dataCommentTiktok.xlsx', index=False)

**1. Cleaning Data**

In [ ]:
import pandas as pd

# load data
df = pd.read_excel('dataCommentTiktok.xlsx')

# ambil kolom penting
df = df[['text', 'videoWebUrl']]

# hapus data kosong & duplikat
df = df.dropna(subset=['text'])
df = df.drop_duplicates()

df.head()

**2. Preprocessing**

In [ ]:
import re

# Fungsi untuk menghapus karakter berulang
def remove_repeated_chars(text):
    return re.sub(r'(.)\1+', r'\1\1', text) # Membatasi maksimal 2 karakter kembar

def clean_text(text):
    text = str(text).lower() # Pastikan jadi string dan huruf kecil
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Mapping normalisasi
normalisasi_dict = {
    "gk": "tidak", "ga": "tidak", "nggak": "tidak",
    "bgt": "banget", "bgtt": "banget", "bgus": "bagus"
}

def normalize_text(text):
    words = text.split()
    normalized = [normalisasi_dict.get(word, word) for word in words]
    return " ".join(normalized)

In [ ]:
# Jalankan secara berurutan
df['clean_text'] = df['text'].apply(clean_text)
df['clean_text'] = df['clean_text'].apply(remove_repeated_chars)
df['clean_text'] = df['clean_text'].apply(normalize_text)

**3. Labeling (SENTIMENT)**

In [ ]:
from transformers import pipeline

# 1. Panggil Modelnya
classifier = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

# 2. Buat fungsi ambil label saja
def get_label(text):
    if not text or text.strip() == "":
        return "neutral"
    try:
        hasil = classifier(text)
        return hasil[0]['label']
    except:
        return "neutral"

# 3. Terapkan ke kolom clean_text (Ini yang membuat kolom 'sentiment')
print("Sedang memproses sentimen, mohon tunggu...")
df['sentiment'] = df['clean_text'].apply(get_label)

# 4. Cek apakah kolom sudah muncul sekarang
print("Kolom sekarang:", df.columns.tolist())
print(df[['clean_text', 'sentiment']].head())

**4. Visualization**

In [ ]:
# sentiment overall
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar')
plt.title("Sentiment Analysis MBG (TikTok)")
plt.xlabel("Sentiment")
plt.ylabel("Jumlah Komentar")
plt.show()

In [ ]:
# sentiment per-vidio
import matplotlib.pyplot as plt
import pandas as pd

# Buat label Video
df['video_label'] = 'Video ' + (df['videoWebUrl'].factorize()[0] + 1).astype(str)

# Hanya lanjut jika kolom sentiment ada
if 'sentiment' in df.columns:
    # Pivot data: jumlah komentar per sentiment tiap video
    video_sentiment = df.groupby(['video_label','sentiment']).size().unstack(fill_value=0)

    # Plot stacked bar
    video_sentiment.plot(kind='bar', stacked=True, figsize=(12,6))
    plt.title("Analisis Sentimen per Video TikTok")
    plt.xlabel("Nomor Video")
    plt.ylabel("Jumlah Komentar")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Kolom 'sentiment' tidak ditemukan!", df.columns.tolist())

In [ ]:
!pip install wordcloud
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

# Daftar kata yang ingin dibuang (tambah sesuai kebutuhan)
stop_words_id = set(["yg", "yang", "di", "dan", "itu", "ini", "ada", "ke", "dari", "adalah", "untuk", "ga", "gak", "tidak"])
stop_words_id.update(STOPWORDS)

In [ ]:
# Filter data negatif
negatif_text = " ".join(df[df['sentiment'] == 'negative']['clean_text'])

if negatif_text:
    wordcloud_neg = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='Reds', # Warna merah agar kontras untuk negatif
        stopwords=stop_words_id
    ).generate(negatif_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_neg, interpolation='bilinear')
    plt.title('Kumpulan Kata - Sentimen NEGATIF', fontsize=15)
    plt.axis('off')
    plt.show()
else:
    print("Tidak ada data dengan sentimen negatif.")

In [ ]:
# Filter data positif
positif_text = " ".join(df[df['sentiment'] == 'positive']['clean_text'])

if positif_text:
    wordcloud_pos = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis', # Warna hijau-biru
        stopwords=stop_words_id
    ).generate(positif_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_pos, interpolation='bilinear')
    plt.title('Kumpulan Kata - Sentimen POSITIF', fontsize=15)
    plt.axis('off')
    plt.show()
else:
    print("Tidak ada data dengan sentimen positif.")

In [ ]:
# 1. Filter data netral
netral_text = " ".join(df[df['sentiment'] == 'neutral']['clean_text'])

# 2. Cek apakah teks netral ada isinya
if netral_text:
    # Buat WordCloud dengan warna abu-abu/biru tenang (colormap='bone' atau 'cool')
    wordcloud_neu = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='cool', # Warna biru-cyan yang tenang
        stopwords=stop_words_id
    ).generate(netral_text)

    # 3. Visualisasi
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_neu, interpolation='bilinear')
    plt.title('Kumpulan Kata - Sentimen NETRAL', fontsize=15)
    plt.axis('off')
    plt.show()
else:
    print("Tidak ada data dengan sentimen netral.")

In [ ]:
df.to_excel('final_mbg_analysis.xlsx', index=False)

In [ ]:
from sklearn.model_selection import train_test_split

# bagi 80% data untuk latihan, 20% untuk ujian
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% data uji
    random_state=42
)

print("Data berhasil dibagi!")
print(f"Jumlah data training: {len(X_train)}")
print(f"Jumlah data testing: {len(X_test)}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Training Naive Bayes
model_nb = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])
model_nb.fit(X_train, y_train)
y_pred_nb = model_nb.predict(X_test)

# 2. Training Random Forest
model_rf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

# Cetak Hasil
print(f"Akurasi Naive Bayes: {accuracy_score(y_test, y_pred_nb):.2%}")
print(f"Akurasi Random Forest: {accuracy_score(y_test, y_pred_rf):.2%}")

Akurasi Naive Bayes: 63.76%
Akurasi Random Forest: 62.42%


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Lihat hasil untuk Naive Bayes (karena akurasinya lebih tinggi)
cm = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=model_nb.classes_,
            yticklabels=model_nb.classes_)
plt.title('Confusion Matrix - Naive Bayes')
plt.xlabel('Tebakan Model')
plt.ylabel('Kenyataan Asli')
plt.show()

In [ ]:
print(df['sentiment'].value_counts())

In [ ]:
from transformers import pipeline

# Model sentiment Bahasa Indonesia
classifier = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-sentiment-classifier")

while True:
    teks = input("Komentar (ketik 'keluar' untuk stop): ").strip()
    if teks.lower() == "keluar": break
    if not teks: continue
    label = classifier(teks)[0]['label'].lower()
    print("Hasil:", "✅ POSITIF" if label=='positive' else "❌ NEGATIF" if label=='negative' else "⚪ NETRAL")
    print("-"*30)